In [7]:
import os
import pandas as pd
from openpyxl import load_workbook, Workbook
from datetime import datetime

# Configuration paths

INPUT_EXCEL = r"C:\Users\phansanti\OneDrive - Agoda\Invoice\ClickUP invoices\generate_invoice_data.xlsx"
OUTPUT_DIR= r"C:\Users\phansanti\OneDrive - Agoda\Invoice\OUT_Invoices"
TEMPLATE_XLSX = r"C:\Users\phansanti\OneDrive - Agoda\Documents\kae_projects\Freelancer-Invoices-Package\Revised workflow\MASTER Expected Invoice.xlsx"

def ensure_output_dir(path: str):
    os.makedirs(path, exist_ok=True)

def _detect_column(df, keywords):
    """Return the first column name matching any of the provided keywords (case‑insensitive)."""
    for col in df.columns:
        lowered = col.lower()
        for kw in keywords:
            if kw in lowered:
                return col
    return None


def load_input_data(path: str) -> pd.DataFrame:
    """Read the raw freelancer job data from the supplied Excel file.

    Expected columns (at minimum):
        - Freelancer (str)
        - Date (datetime or str)
        - Hours (float)
        - Rate (float) – hourly rate
    """
    df = pd.read_excel(path)
    original_cols = list(df.columns)
    # Normalise column names for case‑insensitive matching
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
    # Identify freelancer column (explicitly column E: send_to_users)
    freelancer_col = 'send_to_users' if 'send_to_users' in df.columns else _detect_column(df, ['email', 'send_to_users'])
    # Identify date column (explicitly column J: service_month_date_done)
    date_col = 'service_month_date_done' if 'service_month_date_done' in df.columns else _detect_column(df, ['date_created', 'service_month_date_done'])
    # Identify amount column
    amount_col = _detect_column(df, ['total_formula'])
    # If no explicit amount, compute from rate and wordcount if available
    if amount_col is None:
        rate_col = _detect_column(df, ['vendor_rate_number'])
        wc_col = _detect_column(df, ['wordcount_number'])
        if rate_col and wc_col:
            # Create a temporary amount column by multiplying rate and wordcount
            df['computed_amount'] = pd.to_numeric(df[rate_col], errors='coerce') * pd.to_numeric(df[wc_col], errors='coerce')
            amount_col = 'computed_amount'
    # Validate presence of required columns
    missing = []
    for name, col in [('Freelancer', freelancer_col), ('Date', date_col), ('Amount', amount_col)]:
        if col is None:
            missing.append(name)
    if missing:
        raise ValueError(f"Missing required columns: {', '.join(missing)}. Found columns: {original_cols}")
    # Rename columns to standardized names
    df = df.rename(columns={freelancer_col: 'freelancer', date_col: 'date', amount_col: 'amount'})
    # Drop rows with missing essential data (freelancer, date, amount)
    df = df.dropna(subset=['freelancer', 'date', 'amount'])
    return df


def aggregate_monthly(df: pd.DataFrame) -> pd.DataFrame:
    """Group by freelancer and month, calculate total amount."""
    # Drop rows with missing essential data before aggregation
    df = df.dropna(subset=['freelancer', 'date', 'amount'])
    # Ensure Date column is datetime
    if not pd.api.types.is_datetime64_any_dtype(df['date']):
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['month'] = df['date'].dt.to_period('M')
    # Aggregate total amount per freelancer per month
    agg = df.groupby(['freelancer', 'month']).agg(
        total_amount=pd.NamedAgg(column='amount', aggfunc='sum')
    ).reset_index()
    # Convert month back to string for naming
    agg['month_str'] = agg['month'].dt.strftime('%Y-%m')
    return agg

def create_invoice_file(template_path: str, output_path: str, summary_df: pd.DataFrame, freelancer_name: str):
    """Copy the master template, replace placeholder names, and inject the summary data.

    - Replaces any cell containing the placeholder "Marton" with the actual freelancer name.
    - Writes the aggregated data to a new sheet named 'Summary'.
    """
    wb = load_workbook(template_path)
    # Replace placeholder name throughout the workbook
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for cell in row:
                if isinstance(cell.value, str) and cell.value.strip() == "Marton":
                    cell.value = freelancer_name
    # Remove existing Summary sheet if present
    if 'Summary' in wb.sheetnames:
        ws = wb['Summary']
        wb.remove(ws)
    ws = wb.create_sheet('Summary')
    # Write header
    headers = ['Freelancer', 'Month', 'Total Amount']
    ws.append(headers)
    # Write rows
    for _, row in summary_df.iterrows():
        ws.append([
            row['freelancer'],
            row['month_str'],
            round(row['total_amount'], 2)
        ])
    wb.save(output_path)
    """Copy the master template and inject the summary data.

    The function writes the aggregated data to a new sheet named 'Summary'.
    """
    wb = load_workbook(template_path)
    # Remove existing Summary sheet if present
    if 'Summary' in wb.sheetnames:
        ws = wb['Summary']
        wb.remove(ws)
    ws = wb.create_sheet('Summary')
    # Write header
    headers = ['Freelancer', 'Month', 'Total Amount']
    ws.append(headers)
    # Write rows
    for _, row in summary_df.iterrows():
        ws.append([
            row['freelancer'],
            row['month_str'],
            round(row['total_amount'], 2)
        ])
    wb.save(output_path)

def main():
    ensure_output_dir(OUTPUT_DIR)
    try:
        df = load_input_data(INPUT_EXCEL)
        summary = aggregate_monthly(df)
        # Build a mapping of extra info (e.g., invoice_number) per freelancer per month
        df['month'] = df['date'].dt.to_period('M')
        extra_info = df.groupby(['freelancer', 'month']).agg(
            invoice_number=pd.NamedAgg(column='invoice_number', aggfunc='first')
        ).reset_index()
        # Merge extra info with summary
        merged = summary.merge(extra_info, on=['freelancer', 'month'], how='left')
        # Generate an invoice for each freelancer/month pair
        for _, row in merged.iterrows():
            freelancer = row['freelancer']
            month_str = row['month_str']
            inv_num = row['invoice_number'] if pd.notna(row['invoice_number']) else ''
            # Clean up filename components
            safe_freelancer = freelancer.replace(' ', '_')
            safe_inv = str(inv_num).replace(' ', '_')
            filename = f"Invoice_{safe_freelancer}_{month_str}"
            if safe_inv:
                filename += f"_{safe_inv}"
            output_file = os.path.join(OUTPUT_DIR, f"{filename}.xlsx")
            # Pass freelancer name for placeholder replacement
            create_invoice_file(TEMPLATE_XLSX, output_file, pd.DataFrame([row]), freelancer)
            print(f"Generated invoice for {freelancer} {month_str}: {output_file}")
    except Exception as e:
        print(f"Error during invoice generation: {e}")

if __name__ == "__main__":
    main()


Error during invoice generation: [Errno 2] No such file or directory: 'C:\\Users\\phansanti\\OneDrive - Agoda\\Invoice\\ClickUP invoices\\generate_invoice_data.xlsx'


In [9]:
import os
print(os.path.exists(
    r"C:\Users\phansanti\OneDrive - Agoda\Invoice\ClickUP invoices\generate_invoice_data.xlsx"
))

False
